# 8. Train and compare yoga subtype classifiers

Target: `subpose_label`, including left/right variants. Features remain keypoints and angles only; subtype IDs and labels are never input features. Run notebook 7 before 8.

Nine subtypes occur in training. The current test set additionally contains goddess_subpose_3 and plank_subpose_3 (four samples total). Full test metrics include these samples as errors, with all 11 test labels in the confusion matrix. Validation uses nine labels. A separate known-subtype test report covers the remaining 461 samples. No test samples are moved into training.

Outputs include saved models, subtype prediction scores, per-subtype reports, confusion matrices (PNG/CSV), label coverage, manifests and validation rankings. Notebook 8 compares all six classifiers on the same subtype task.

Model formats: code 7 uses joblib pipelines; code 8 uses native XGBoost JSON and LightGBM TXT plus label_mapping.json. LightGBM 4.6.0 is used in this project.

In [ ]:
from pathlib import Path
import json
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
TARGET_COLUMN = 'subpose_label'
SAMPLE_COLUMN = 'image_name'
SELECTION_METRIC = 'macro_f1'


def find_project_root(start: Path = Path.cwd()) -> Path:
    for directory in (start.resolve(), *start.resolve().parents):
        candidate = directory / 'csv_data' / 'prepared_to_train' / 'keypoints_train.csv'
        if candidate.is_file():
            return directory
    raise FileNotFoundError('Could not find csv_data/prepared_to_train/keypoints_train.csv')


PROJECT_ROOT = find_project_root()
INPUT_DIR = PROJECT_ROOT / 'csv_data' / 'prepared_to_train'
OUTPUT_DIR = PROJECT_ROOT / 'model' / 'code8_boosting_subpose_classification'
MODEL_DIR = OUTPUT_DIR / 'models'
PREDICTION_DIR = OUTPUT_DIR / 'predictions'
CONFUSION_DIR = OUTPUT_DIR / 'confusion_matrices'
for directory in (OUTPUT_DIR, MODEL_DIR, PREDICTION_DIR, CONFUSION_DIR):
    directory.mkdir(parents=True, exist_ok=True)

INPUT_FILES = {
    'train': INPUT_DIR / 'keypoints_train.csv',
    'validation': INPUT_DIR / 'keypoints_validation.csv',
    'test': INPUT_DIR / 'keypoints_test_dbscan_subposes.csv',
}
PROJECT_ROOT

import sys
sys.path.insert(0, str(PROJECT_ROOT / '.dependencies'))
from xgboost import XGBClassifier
import lightgbm
from lightgbm import LGBMClassifier, Booster
from sklearn.preprocessing import LabelEncoder
import hashlib
import importlib.metadata

BASELINE_DIR = PROJECT_ROOT / 'model' / 'code7_subpose_classification'


In [ ]:
datasets = {split: pd.read_csv(path) for split, path in INPUT_FILES.items()}
FEATURE_COLUMNS = [
    column for column in datasets['train'].columns
    if column.startswith('kp_') or column.endswith('_angle_deg')
]
if not FEATURE_COLUMNS:
    raise ValueError('No keypoint or angle feature columns were found.')

train_columns = set(datasets['train'].columns)
for split, dataframe in datasets.items():
    missing = {SAMPLE_COLUMN, TARGET_COLUMN, *FEATURE_COLUMNS}.difference(dataframe.columns)
    if missing:
        raise ValueError(f'{split} is missing columns: {sorted(missing)}')
    if dataframe[[SAMPLE_COLUMN, TARGET_COLUMN]].isna().any().any():
        raise ValueError(f'{split} has missing sample IDs or subtype labels.')
    values = dataframe[FEATURE_COLUMNS].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError(f'{split} feature data contains NaN or infinite values.')
    if dataframe[SAMPLE_COLUMN].duplicated().any():
        raise ValueError(f'{split} contains duplicate sample paths.')

CLASSES = sorted(datasets['train'][TARGET_COLUMN].astype(str).unique())
for split in ('validation', 'test'):
    unknown = set(datasets[split][TARGET_COLUMN].astype(str)).difference(CLASSES)
    if unknown:
        if split == 'validation':
            raise ValueError(f'Validation contains subtypes absent from training: {sorted(unknown)}')
        print(f'{split}: unseen training subtypes retained for evaluation: {sorted(unknown)}')

X = {split: df[FEATURE_COLUMNS] for split, df in datasets.items()}
y = {split: df[TARGET_COLUMN].astype(str) for split, df in datasets.items()}
pd.DataFrame({
    split: y[split].value_counts().reindex(CLASSES, fill_value=0)
    for split in datasets
}).rename_axis('pose')


for first, second in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    if set(datasets[first][SAMPLE_COLUMN]) & set(datasets[second][SAMPLE_COLUMN]):
        raise ValueError(f'Sample overlap between {first} and {second}')
EVALUATION_CLASSES = {split: sorted(set(CLASSES) | set(df[TARGET_COLUMN].astype(str)))
                      for split, df in datasets.items()}
coverage = pd.DataFrame({split: df[TARGET_COLUMN].value_counts()
                         for split, df in datasets.items()}).fillna(0).astype(int)
coverage['seen_in_training'] = coverage.index.isin(CLASSES)
coverage.rename_axis('subpose').to_csv(OUTPUT_DIR / 'subtype_split_coverage.csv')
unseen_tables = []
for split, df in datasets.items():
    unseen = df.loc[~df[TARGET_COLUMN].isin(CLASSES), [SAMPLE_COLUMN, 'label', TARGET_COLUMN]].copy()
    unseen.insert(0, 'split', split)
    unseen_tables.append(unseen)
pd.concat(unseen_tables).to_csv(OUTPUT_DIR / 'unseen_subtype_samples.csv', index=False)
display(coverage)

encoder = LabelEncoder().fit(CLASSES)
y_encoded = encoder.transform(y['train'])
for first, second in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    if set(datasets[first][SAMPLE_COLUMN]) & set(datasets[second][SAMPLE_COLUMN]):
        raise ValueError(f'Sample overlap between {first} and {second}')
baseline_manifest = json.loads((BASELINE_DIR / 'model_manifest.json').read_text(encoding='utf-8'))
if baseline_manifest['target_column'] != TARGET_COLUMN:
    raise ValueError('Run notebook 7 for subtype classification first.')
for split, path in INPUT_FILES.items():
    if baseline_manifest['input_sha256'][split] != hashlib.sha256(path.read_bytes()).hexdigest():
        raise ValueError('Baseline input data differs from current data.')
if baseline_manifest['feature_columns'] != FEATURE_COLUMNS or baseline_manifest['classes'] != CLASSES:
    raise ValueError('Baseline features/classes differ from the current data.')
baseline_metrics = pd.read_csv(BASELINE_DIR / 'overall_metrics.csv')
for row in baseline_metrics.itertuples():
    previous = pd.read_csv(BASELINE_DIR / 'predictions' / f'{row.model}_{row.split}_predictions.csv')
    current = datasets[row.split].set_index(SAMPLE_COLUMN)[TARGET_COLUMN].astype(str)
    if previous['sample'].duplicated().any() or set(previous['sample']) != set(current.index):
        raise ValueError('Baseline evaluation samples differ from current samples.')
    if not np.array_equal(previous['true_label'].astype(str), current.loc[previous['sample']].to_numpy()):
        raise ValueError('Baseline evaluation labels differ from current labels.')
    measured_f1 = precision_recall_fscore_support(previous.true_label, previous.predicted_label,
        labels=EVALUATION_CLASSES[row.split], average='macro', zero_division=0)[2]
    if not np.isclose(measured_f1, row.macro_f1):
        raise ValueError('Baseline metrics do not match saved predictions.')


In [ ]:
models = {
    'xgboost': XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, objective='multi:softprob',
        eval_metric='mlogloss', tree_method='hist', n_jobs=4, random_state=RANDOM_STATE),
    'lightgbm': LGBMClassifier(n_estimators=500, num_leaves=15, learning_rate=0.05,
        subsample=0.9, subsample_freq=1, colsample_bytree=0.9, verbosity=-1,
        objective='multiclass', n_jobs=4, random_state=RANDOM_STATE),
}


In [ ]:
def safe_name(value: str) -> str:
    return re.sub(r'[^0-9A-Za-z_-]+', '_', str(value)).strip('_')


def aligned_probabilities(model, features: pd.DataFrame) -> np.ndarray:
    raw = model.predict_proba(features)
    model_classes = encoder.inverse_transform(model.classes_.astype(int)).tolist()
    indices = [model_classes.index(pose) for pose in CLASSES]
    return raw[:, indices]


def save_confusion_figure(matrix: np.ndarray, title: str, path: Path, labels) -> None:
    fig, ax = plt.subplots(figsize=(14, 12))
    image = ax.imshow(matrix, cmap='Blues')
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    ax.set(xticks=range(len(labels)), yticks=range(len(labels)),
           xticklabels=labels, yticklabels=labels,
           xlabel='Predicted subtype', ylabel='True subtype', title=title)
    plt.setp(ax.get_xticklabels(), rotation=35, ha='right')
    threshold = matrix.max() / 2 if matrix.size else 0
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            ax.text(column, row, int(matrix[row, column]), ha='center', va='center',
                    color='white' if matrix[row, column] > threshold else 'black')
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches='tight')
    plt.close(fig)


overall_rows = []
pose_rows = []
confusion_rows = []
fit_rows = []
known_rows = []

for model_name, model in models.items():
    print(f'Training {model_name} ...')
    started = time.perf_counter()
    with warnings.catch_warnings():
        warnings.filterwarnings('always', category=ConvergenceWarning)
        model.fit(X['train'], y_encoded)
    fit_seconds = time.perf_counter() - started
    model_path = MODEL_DIR / ('xgboost.json' if model_name == 'xgboost' else 'lightgbm.txt')
    if model_name == 'xgboost':
        model.save_model(str(MODEL_DIR / 'xgboost.json'))
    else:
        model.booster_.save_model(str(MODEL_DIR / 'lightgbm.txt'))
    fit_rows.append({'model': model_name, 'fit_seconds': fit_seconds, 'model_file': str(model_path)})

    for split in ('validation', 'test'):
        evaluation_labels = EVALUATION_CLASSES[split]
        predicted = encoder.inverse_transform(model.predict(X[split]).astype(int))
        probabilities = aligned_probabilities(model, X[split])
        precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
            y[split], predicted, labels=evaluation_labels, average='macro', zero_division=0
        )
        precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
            y[split], predicted, labels=evaluation_labels, average='weighted', zero_division=0
        )
        overall_rows.append({
            'model': model_name, 'split': split, 'samples': len(y[split]), 'evaluated_class_count': len(evaluation_labels),
            'unseen_subtype_samples': int((~y[split].isin(CLASSES)).sum()),
            'accuracy': accuracy_score(y[split], predicted),
            'macro_precision': precision_macro, 'macro_recall': recall_macro,
            'macro_f1': f1_macro, 'weighted_precision': precision_weighted,
            'weighted_recall': recall_weighted, 'weighted_f1': f1_weighted,
        })

        report = classification_report(
            y[split], predicted, labels=evaluation_labels, target_names=evaluation_labels,
            output_dict=True, zero_division=0,
        )
        for pose in evaluation_labels:
            pose_rows.append({
                'model': model_name, 'split': split, 'subpose': pose, 'seen_in_training': pose in CLASSES,
                'precision': report[pose]['precision'], 'recall': report[pose]['recall'],
                'f1': report[pose]['f1-score'], 'support': int(report[pose]['support']),
            })

        matrix = confusion_matrix(y[split], predicted, labels=evaluation_labels)
        pd.DataFrame(matrix, index=evaluation_labels, columns=evaluation_labels).rename_axis('true_subpose').to_csv(CONFUSION_DIR / f'{model_name}_{split}_confusion_matrix.csv')
        for true_index, true_pose in enumerate(evaluation_labels):
            for predicted_index, predicted_pose in enumerate(evaluation_labels):
                confusion_rows.append({
                    'model': model_name, 'split': split, 'true_subpose': true_pose,
                    'predicted_subpose': predicted_pose,
                    'count': int(matrix[true_index, predicted_index]),
                })
        save_confusion_figure(
            matrix, f'{model_name} - {split}',
            CONFUSION_DIR / f'{model_name}_{split}_confusion_matrix.png', evaluation_labels,
        )

        prediction_table = pd.DataFrame({
            'sample': datasets[split][SAMPLE_COLUMN].astype(str),
            'true_label': y[split], 'predicted_label': predicted,
            'is_correct': y[split].to_numpy() == predicted,
            'true_subtype_seen_in_training': y[split].isin(CLASSES),
            'main_pose': datasets[split]['label'],
        })
        for class_index, pose in enumerate(CLASSES):
            prediction_table[f'score_{safe_name(pose)}'] = probabilities[:, class_index]
        prediction_table.to_csv(
            PREDICTION_DIR / f'{model_name}_{split}_predictions.csv', index=False
        )

        if split == 'test':
            known_mask = y[split].isin(CLASSES).to_numpy()
            known_truth, known_predicted = y[split][known_mask], predicted[known_mask]
            kp, kr, kf, _ = precision_recall_fscore_support(known_truth, known_predicted,
                labels=CLASSES, average='macro', zero_division=0)
            known_rows.append({'model': model_name, 'split': 'test_known_subtypes',
                'samples': int(known_mask.sum()), 'excluded_unseen_samples': int((~known_mask).sum()),
                'accuracy': accuracy_score(known_truth, known_predicted),
                'macro_precision': kp, 'macro_recall': kr, 'macro_f1': kf})

overall_metrics = pd.DataFrame(overall_rows).sort_values(['split', 'macro_f1'], ascending=[True, False])
per_pose_metrics = pd.DataFrame(pose_rows).sort_values(['split', 'model', 'subpose'])
confusion_long = pd.DataFrame(confusion_rows)
training_times = pd.DataFrame(fit_rows).sort_values('fit_seconds')

overall_metrics.to_csv(OUTPUT_DIR / 'overall_metrics.csv', index=False)
per_pose_metrics.to_csv(OUTPUT_DIR / 'classification_report_per_subpose.csv', index=False)
confusion_long.to_csv(OUTPUT_DIR / 'confusion_matrices.csv', index=False)
training_times.to_csv(OUTPUT_DIR / 'training_times.csv', index=False)
display(overall_metrics)
display(per_pose_metrics)

pd.DataFrame(known_rows).to_csv(OUTPUT_DIR / 'known_subtype_test_metrics.csv', index=False)


In [ ]:
validation_ranking = (
    overall_metrics[overall_metrics['split'].eq('validation')]
    .sort_values([SELECTION_METRIC, 'accuracy'], ascending=False)
    .reset_index(drop=True)
)
best_model_name = validation_ranking.loc[0, 'model']
best_model = models[best_model_name]
best_model_path = MODEL_DIR / ('xgboost.json' if best_model_name == 'xgboost' else 'lightgbm.txt')

manifest = {
    'best_model': best_model_name,
    'selection_split': 'validation',
    'selection_metric': SELECTION_METRIC,
    'selection_value': float(validation_ranking.loc[0, SELECTION_METRIC]),
    'target_column': TARGET_COLUMN,
    'sample_column': SAMPLE_COLUMN,
    'classes': CLASSES,
    'evaluation_classes': EVALUATION_CLASSES,
    'input_sha256': {split: hashlib.sha256(path.read_bytes()).hexdigest() for split, path in INPUT_FILES.items()},
    'evaluation_note': 'Full test includes unseen subtypes as errors; macro metrics use the union of training classes and split labels. Known-subtype test metrics are separate.',
    'encoded_class_mapping': {str(i): pose for i, pose in enumerate(CLASSES)},
    'feature_columns': FEATURE_COLUMNS,
    'input_files': {key: str(value) for key, value in INPUT_FILES.items()},
    'best_model_file': str(best_model_path.relative_to(OUTPUT_DIR)),
    'model_files': {'xgboost': 'models/xgboost.json', 'lightgbm': 'models/lightgbm.txt'},
    'label_mapping_file': 'label_mapping.json',
    'note': 'Model selection used validation macro-F1 only; test was not used to select a model.',
}
with (OUTPUT_DIR / 'model_manifest.json').open('w', encoding='utf-8') as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)
validation_ranking.to_csv(OUTPUT_DIR / 'validation_model_ranking.csv', index=False)

required_outputs = [
    best_model_path, OUTPUT_DIR / 'overall_metrics.csv',
    OUTPUT_DIR / 'classification_report_per_subpose.csv',
    OUTPUT_DIR / 'confusion_matrices.csv', OUTPUT_DIR / 'model_manifest.json',
]
assert all(path.is_file() for path in required_outputs)
assert all((MODEL_DIR / filename).is_file() for filename in ('xgboost.json', 'lightgbm.txt'))
assert len(list(PREDICTION_DIR.glob('*_predictions.csv'))) == 2 * len(models)
assert len(list(CONFUSION_DIR.glob('*.png'))) == 2 * len(models)
assert np.allclose(
    pd.read_csv(PREDICTION_DIR / f'{best_model_name}_test_predictions.csv')
      [[f'score_{safe_name(pose)}' for pose in CLASSES]].sum(axis=1),
    1.0,
)

print(f'Best model: {best_model_name}')
print(f'Validation macro-F1: {validation_ranking.loc[0, SELECTION_METRIC]:.4f}')
print(f'All outputs saved to: {OUTPUT_DIR}')
display(validation_ranking)


In [ ]:
# Select using validation only; the test scores are descriptive held-out results.
(OUTPUT_DIR / 'label_mapping.json').write_text(json.dumps({'classes': CLASSES, 'encoded_class_mapping': {str(i): pose for i, pose in enumerate(CLASSES)}}, indent=2), encoding='utf-8')
combined = pd.concat([baseline_metrics, overall_metrics], ignore_index=True)
combined.to_csv(OUTPUT_DIR / 'comparison_all_models.csv', index=False)
pd.concat([pd.read_csv(BASELINE_DIR / 'classification_report_per_subpose.csv'), per_pose_metrics],
          ignore_index=True).to_csv(OUTPUT_DIR / 'comparison_per_subpose.csv', index=False)
ranking = combined[combined.split.eq('validation')].sort_values(
    ['macro_f1', 'accuracy'], ascending=False).reset_index(drop=True)
ranking.to_csv(OUTPUT_DIR / 'comparison_validation_ranking.csv', index=False)
baseline_best = baseline_metrics[baseline_metrics.split.eq('validation')].sort_values(
    ['macro_f1', 'accuracy'], ascending=False).iloc[0]['model']
deltas = []
for row in overall_metrics.to_dict('records'):
    reference = baseline_metrics[(baseline_metrics.model == baseline_best) &
                                 (baseline_metrics.split == row['split'])].iloc[0]
    deltas.append({'model': row['model'], 'split': row['split'], 'baseline': baseline_best,
        'accuracy_delta_percentage_points': 100 * (row['accuracy'] - reference.accuracy),
        'macro_f1_delta_percentage_points': 100 * (row['macro_f1'] - reference.macro_f1)})
pd.DataFrame(deltas).to_csv(OUTPUT_DIR / 'comparison_vs_best_baseline.csv', index=False)
table = combined.pivot(index='model', columns='split', values=['accuracy', 'macro_f1'])
table = table.reindex(ranking.model)
table.to_csv(OUTPUT_DIR / 'comparison_summary.csv')
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, metric in zip(axes, ['accuracy', 'macro_f1']):
    table[metric].plot.bar(ax=ax, ylim=(0, 1.05), rot=30, title=metric)
    ax.set_ylabel('Score')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'comparison_all_models.png', dpi=180, bbox_inches='tight')
plt.close(fig)
manifest.update({
    'input_sha256': {split: hashlib.sha256(path.read_bytes()).hexdigest() for split, path in INPUT_FILES.items()},
    'versions': {name: (lightgbm.__version__ if name == 'lightgbm' else importlib.metadata.version(name)) for name in
                 ['xgboost', 'lightgbm', 'scikit-learn', 'numpy', 'pandas']},
    'parameters': {name: model.get_params() for name, model in models.items()},
    'best_overall_by_validation': ranking.iloc[0]['model'],
    'comparison_note': 'Baseline saved evaluation samples and labels verified. Training, validation and test input hashes verified against the subtype baseline manifest.',
})
(OUTPUT_DIR / 'model_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
lines = ['# Subtype classifier comparison', '', 'Models ranked by validation macro-F1.', '',
         '| Model | Validation accuracy | Validation macro-F1 | Test accuracy | Test macro-F1 |',
         '|---|---:|---:|---:|---:|']
for name in ranking.model:
    lines.append('| ' + name + ' | ' + ' | '.join(f'{table.loc[name, (metric, split)]:.4f}'
        for split in ['validation', 'test'] for metric in ['accuracy', 'macro_f1']) + ' |')
lines += ['', f'Best overall by validation: {ranking.iloc[0]["model"]}.',
          f'Best new boosting model by validation: {best_model_name}.', '',
          'Fixed hyperparameters; training uses only the training split. Test scores were not used for selection.',
          manifest['comparison_note'], '', manifest['evaluation_note'], '',
          'The test set contains four samples from two subtypes absent from training; see unseen_subtype_samples.csv and comparison_known_subtype_test.csv.', '',
          'Load XGBoost with XGBClassifier.load_model and LightGBM with Booster(model_file=...). LightGBM Booster.predict returns probabilities; take argmax for class IDs. Decode IDs with the classes array in label_mapping.json. Select input columns in model_manifest.json order.']
(OUTPUT_DIR / 'comparison_report.md').write_text('\n'.join(lines), encoding='utf-8')
for name in models:
    if name == 'xgboost':
        restored = XGBClassifier()
        restored.load_model(str(MODEL_DIR / 'xgboost.json'))
        restored_probabilities = restored.predict_proba(X['test'])
    else:
        restored = Booster(model_file=str(MODEL_DIR / 'lightgbm.txt'))
        restored_probabilities = restored.predict(X['test'])
    np.testing.assert_allclose(restored_probabilities, models[name].predict_proba(X['test']), rtol=1e-6, atol=1e-8)
    assert np.array_equal(restored_probabilities.argmax(axis=1), models[name].predict(X['test']))
    for split in ['validation', 'test']:
        saved = pd.read_csv(PREDICTION_DIR / f'{name}_{split}_predictions.csv')
        assert len(saved) == len(y[split])
        assert np.allclose(saved.filter(like='score_').sum(axis=1), 1.0)
        matrix = pd.read_csv(CONFUSION_DIR / f'{name}_{split}_confusion_matrix.csv', index_col=0)
        assert matrix.to_numpy().sum() == len(y[split])
display(table)
print('Comparison and reload checks passed. Outputs:', OUTPUT_DIR)

pd.concat([pd.read_csv(BASELINE_DIR / 'known_subtype_test_metrics.csv'), pd.read_csv(OUTPUT_DIR / 'known_subtype_test_metrics.csv')], ignore_index=True).to_csv(OUTPUT_DIR / 'comparison_known_subtype_test.csv', index=False)
